In [2]:
import os
from dotenv import load_dotenv
from groq import Groq
from pydantic import create_model
import inspect, json
from inspect import Parameter

print(load_dotenv())

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

True


## **Custom agent**

**Define the functions**

In [3]:
def abc(num1:int, num2:int)->int:
    "Compute abc between two numbers"
    return 2*(num1) - 2*(num2)

In [4]:
abc(2, 3)

-2

In [5]:
def jsonschema(f):
    """
    Generate a JSON schema for the input parameters of the given function.

    Parameters:
        f (FunctionType): The function for which to generate the JSON schema.

    Returns:
        Dict: A dictionary containing the function name, description, and parameters schema.
    """
    kw = {n: (o.annotation, ... if o.default == Parameter.empty else o.default)
            for n, o in inspect.signature(f).parameters.items()}
    s = create_model(f'Input for `{f.__name__}`', **kw).schema()
    return dict(name=f.__name__, description=f.__doc__, parameters=s)

In [6]:
abc_json = jsonschema(abc)
abc_json

{'name': 'abc',
 'description': 'Compute abc between two numbers',
 'parameters': {'properties': {'num1': {'title': 'Num1', 'type': 'integer'},
   'num2': {'title': 'Num2', 'type': 'integer'}},
  'required': ['num1', 'num2'],
  'title': 'Input for `abc`',
  'type': 'object'}}

In [7]:
model_name = "llama-3.3-70b-versatile"

**Ask Groq**

In [8]:
client = Groq()

response = client.chat.completions.create(
  model= model_name,
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "compute abc between 2 and 3"},
  ],
)

In [9]:
response.choices[0].message.content

'To compute the absolute difference, also known as the absolute value of the difference, between 2 and 3:\n\n|2 - 3| = |-1| = 1\n\nSo the absolute difference between 2 and 3 is 1.'

In [15]:
messages= [
    {"role": "user", "content": "Compute abc between 2 and 3"}
]

# Pass th function to groq model
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    functions=[abc_json],
    function_call="auto",
    temperature=0
)

In [11]:
response

ChatCompletion(id='chatcmpl-e37ef9c8-5015-4811-91c8-6373640bb737', choices=[Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=FunctionCall(arguments='{"num1":2,"num2":3}', name='abc'), reasoning=None, tool_calls=None))], created=1778583530, model='llama-3.3-70b-versatile', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_ce7bc1685b', usage=CompletionUsage(completion_tokens=21, prompt_tokens=237, total_tokens=258, completion_time=0.047009754, completion_tokens_details=None, prompt_time=0.013427763, prompt_tokens_details=None, queue_time=0.048241546, total_time=0.060437517), usage_breakdown=None, x_groq=XGroq(id='req_01krdxdtkmeghva26epfsabw0m', debug=None, seed=1357225914, usage=None))

**Executing the function by extracting the info from the output of the model**

In [12]:
print(response.choices[0].message.function_call)
print(response.choices[0].message.function_call.arguments)
print(type(response.choices[0].message.function_call.arguments))

FunctionCall(arguments='{"num1":2,"num2":3}', name='abc')
{"num1":2,"num2":3}
<class 'str'>


In [13]:
func_name = response.choices[0].message.function_call.name
func_args = json.loads(response.choices[0].message.function_call.arguments)
print("Function name:", func_name)
print("Function arguments:", func_args)
print(type(func_args))

Function name: abc
Function arguments: {'num1': 2, 'num2': 3}
<class 'dict'>


In [14]:
if func_name == 'abc':
    result = abc(**func_args)
print(result)

-2


## **Using Langchain**

In [16]:
from langchain_core.tools import tool

@tool
def abc(num1:int, num2:int)->int:
    "Compute abc between two numbers"
    return 2*(num1) - 2*(num2)

In [17]:
abc.description

'Compute abc between two numbers'

In [18]:
from langchain_groq import ChatGroq 
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [19]:
tools = [abc]

llm_with_tools = llm.bind_tools(tools)

In [20]:
response = llm_with_tools.invoke("Compute abc between 2 and 3")

In [21]:
response.additional_kwargs

{'tool_calls': [{'id': 'rjae3mfkg',
   'function': {'arguments': '{"num1":2,"num2":3}', 'name': 'abc'},
   'type': 'function'}]}